# Sofifa Scraping

In [6]:
import csv
import time
import random
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# REPLACE THIS with your massive URL containing all 46 columns
BASE_URL = "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCol%5B%5D=gh&showCol%5B%5D=gc&showCol%5B%5D=gp&showCol%5B%5D=gr"
CSV_FILENAME = "data/sofifa/newdata/sofifa_players.csv"
TOTAL_PLAYERS = 400000 
PLAYERS_PER_PAGE = 60

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        # SoFifa's first column (the avatar picture) has no text in the header.
        # We dynamically rename this header to "ID" for our CSV.
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    return headers

def extract_rows_from_html(soup, limit=None):
    player_data = []
    rows = soup.select("table tbody tr")
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker (ads have very few columns)
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION (Strictly Isolated)
                if 'col-name' in classes:
                    links = td.find_all("a", href=lambda h: h and "/player/" in h)
                    name_text = ""
                    for a in links:
                        # Find the hyperlink that actually has the text of the name, not the avatar image
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        # Fallback just in case
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/player/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER STATS (Including the real ID column)
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
                    
            player_data.append(row_values)
            
            if limit and len(player_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return player_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        # headless=False is required to pass Cloudflare's bot detection
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        print("Launching browser to solve Cloudflare challenge...")
        page.goto(f"{BASE_URL}&offset=0")
        
        try:
            # Wait for the actual data table to appear on screen
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print("Failed to bypass Cloudflare in time. Please try again.")
            browser.close()
            return

        # === RESOURCE BLOCKER (TURBO MODE) ===
        def block_heavy_resources(route):
            # Block images, CSS, fonts, and media. Allow HTML and XHR/Fetch.
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        # Apply the blocker to all future requests in this browser window
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked for extreme speed).\n")

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-PLAYER VALIDATION TEST ---")
            
            # Reload page with Turbo Mode active
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr") 
            
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            players = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, player in enumerate(players):
                player_dict = dict(zip(columns, player))
                print(f"Player {i+1}: {player_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            print(f"--- STARTING FAST PRODUCTION SCRAPE ({TOTAL_PLAYERS} Players) ---")
            
            # Initialize CSV file with headers
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr")
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                
            # Master Loop
            with tqdm(total=TOTAL_PLAYERS, desc="Scraping SoFifa", unit=" players") as pbar:
                for offset in range(0, TOTAL_PLAYERS, PLAYERS_PER_PAGE):
                    url = f"{BASE_URL}&offset={offset}"
                    
                    # Retry logic for network hiccups
                    for attempt in range(3):
                        try:
                            page.goto(url)
                            page.wait_for_selector("table tbody tr", timeout=15000)
                            
                            soup = BeautifulSoup(page.content(), "html.parser")
                            players = extract_rows_from_html(soup)
                            
                            # Append strictly to CSV
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(players)
                            
                            pbar.update(len(players))
                            
                            if len(players) == 0:
                                print("\n[Notice] No more players found.")
                                browser.close()
                                return
                            break 
                            
                        except Exception as e:
                            print(f"\n[Network/Load Error]. Retrying...")
                            time.sleep(5)
                    
                    # Polite delay to prevent rate limits
                    time.sleep(random.uniform(1.5, 3.0))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        # Safely shut down Chromium
        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

In [7]:
run_test()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
Turbo mode activated (Images/CSS blocked for extreme speed).

--- RUNNING 10-PLAYER VALIDATION TEST ---
Successfully Fetched! Extracted 50 Columns.
--------------------------------------------------
Player 1: {'Unknown': '', 'Name': 'E. Kroupi ST CAM', 'Age': '19', 'Overall rating': '78', 'Potential': '86', 'Team & Contract': 'AFC Bournemouth 2025 ~ 2030', 'ID': '277909', 'Birth year': '2006', 'Height': '179cm 5\'10"', 'foot': 'Right', 'Best position': 'ST', 'Growth': '8', 'Value': '€30.5M', 'Wage': '€53K', 'Total attacking': '368', 'Crossing': '57', 'Finishing': '81 +1', 'Heading accuracy': '72', 'Short passing': '76', 'Volleys': '82 +6', 'Total skill': '357', 'Dribbling': '82 +2', 'Curve': '69', 'FK Accuracy': '55', 'Long passing': '70', 'Ball control': '81', 'Total movement': '386', 'Acceleration': '81', 'Sprint speed': '80', 'Agility': '75', 'Total power': '370', 'Shot power': '80 +1', 'Jumping': '85